# HotpotQA Baseline: Contriever + Qwen2.5-7B

Notebook này chạy baseline **không HotFlip**. `facebook/contriever` xếp hạng 10 document của mỗi mẫu HotpotQA và lấy top-2. Qwen2.5-7B đọc context được lấy và trả lời.

Báo cáo gồm câu hỏi, đáp án chuẩn, đáp án LLM, toàn bộ context được truy xuất, cosine score, EM/F1 và xác suất teacher-forced của đáp án chuẩn.

In [ ]:
# Chọn Runtime > Change runtime type > T4 GPU trước khi chạy.
!nvidia-smi

In [ ]:
%cd /content
!if [ -d HotFlip/.git ]; then git -C HotFlip pull --ff-only; else git clone https://github.com/HoangMaizzz/HotFlip.git; fi
%cd /content/HotFlip
!pip -q install -r requirements-colab.txt

## Cấu hình

`NUM_EXAMPLES=10` phù hợp để chạy thử. Sau khi xác nhận pipeline hoạt động, có thể tăng lên 100 hoặc 300.

In [ ]:
NUM_EXAMPLES = 10
TOP_K = 2
SEED = 42
OUTPUT_DIR = "/content/HotFlip/outputs/colab_baseline"
SHUFFLE = False  # False = lấy các mẫu validation đầu tiên, dễ tái lập

In [ ]:
import subprocess
import sys

command = [
    "python", "-m", "hotflip_rag.baseline",
    "--split", "validation",
    "--num-examples", str(NUM_EXAMPLES),
    "--top-k", str(TOP_K),
    "--seed", str(SEED),
    "--load-in-4bit",
    "--output-dir", OUTPUT_DIR,
]
command.append("--shuffle" if SHUFFLE else "--no-shuffle")
print("Running:", " ".join(command), flush=True)
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end="")
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(
        f"Baseline failed with exit code {return_code}. "
        "The complete root-cause traceback is printed directly above."
    )

## Xem bảng kết quả

In [ ]:
import json
import pandas as pd
from IPython.display import display

with open(f"{OUTPUT_DIR}/baseline_aggregate.json", encoding="utf-8") as f:
    aggregate = json.load(f)
print(json.dumps(aggregate, ensure_ascii=False, indent=2))

df = pd.read_csv(f"{OUTPUT_DIR}/baseline_results.csv")
display(df[[
    "question", "gold_answer", "llm_answer", "retrieved_context",
    "em", "f1", "gold_sequence_probability",
    "gold_mean_token_probability"
]])

In [ ]:
# Tải toàn bộ báo cáo về máy (tùy chọn)
import shutil
from google.colab import files

archive = shutil.make_archive("hotpotqa_baseline_results", "zip", OUTPUT_DIR)
files.download(archive)